In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ukb_utils import tables

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/211209_union_rows.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa044.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/211209_union_rows.log


In [3]:
path = 'data/conditional/common/spa_conditional/211111_spa_conditional_ptv_damaging_missense_Standing_height_conditioning_markers.txt'
ht = hl.import_table(path, no_header = True, delimiter = ' ')

2021-12-13 14:24:50 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type str (not specified)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type str (not specified)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type str (not specified)
  Loading field 'f8' as type str (not specified)
  Loading field 'f9' as type str (not specified)
  Loading field 'f10' as type str (not specified)
  Loading field 'f11' as type str (not specified)
  Loading field 'f12' as type str (not specified)
  Loading field 'f13' as type str (not specified)
  Loading field 'f14' as type str (not specified)
  Loading field 'f15' as type str (not specified)
  Loading field 'f16' as type str (not specified)
  Loading field 'f17' as type str (not specified)
  Loading field 'f18' as type str (not s

In [4]:
ht = ht.annotate(variant = hl.delimit([ht.f0, ht.f1, ht.f3, ht.f4], ':'))
ht = ht.key_by(**hl.parse_variant(ht.variant, reference_genome = 'GRCh38'))
print(ht.locus.contig.collect())

2021-12-13 14:24:57 Hail: INFO: Coerced sorted dataset


['chr9']


In [5]:
ht.show()

,,,,,,,,,,,,,,,,,,,,,,,,
f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,variant,locus,alleles
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,locus<GRCh38>,array<str>
"""chr9""","""136493394""","""rs9314867""","""C""","""T""","""161539""","""0.479048550128937""","""1""","""168604""","""-0.0196650022010299""","""0.00381825821810368""","""-1348.84923205567""","""2.60133001458799e-07""","""68591.3593228599""","""83708.0947314623""",NA,NA,NA,NA,NA,"""-1348.84923205567""","""2.60133001458799e-07""","""chr9:136493394:C:T""",chr9:136493394,"[""C"",""T""]"


In [7]:
# load imputed data and whole exome seq
#wes = hl.read_matrix_table("whole_exome_with_dosages.mt") # GRCh38
wes = hl.read_matrix_table('derived/knockouts/211206/ukb_wes_200k_maf0_0.01_chr9_ptv_damaging_missense_ko.mt') # GRCh38
imp = genotypes.get_ukb_imputed_v3_bgen([9]) # GRCh37

2021-12-13 14:25:19 Hail: INFO: Number of BGEN files parsed: 1
2021-12-13 14:25:19 Hail: INFO: Number of samples in BGEN files: 487409
2021-12-13 14:25:19 Hail: INFO: Number of variants across all BGEN files: 4066774


In [8]:
# collect intersect of individuals
wes_sids = wes.s.collect()
imp_sids = imp.s.collect()
overlap = list(set(wes_sids) & set(imp_sids))[1:5] # just 4 indiviudals
wes = wes.filter_cols(hl.literal(set(overlap)).contains(wes.s))
imp = imp.filter_cols(hl.literal(set(overlap)).contains(imp.s))

In [31]:
# liftover and drop non-matching columns
imp = imp.head(2) # just first variant
imp = variants.liftover(imp)
imp = imp.drop('varid','new_locus','new_alleles','old_locus')
imp = imp.annotate_entries(DS = imp.dosage)
imp = imp.select_entries(imp.DS)
wes = wes.drop('eur')

In [6]:
imp = tables.order_cols(imp, wes)
mt = imp.union_rows(wes)

2021-12-09 13:29:44 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'
2021-12-09 13:29:47 Hail: INFO: Coerced sorted dataset


In [11]:
mt.locus.contig == 'chr21'

<BooleanExpression of type bool>

In [59]:
mt.count()

(168, 4)

In [61]:
from hail.matrixtable import MatrixTable

def order_cols(mt: MatrixTable, ref: MatrixTable) -> MatrixTable:
    r""" Re-order MatrixTable by reference MatrixTable
    
    :param mt: MatrixTable with columns to be re-ordered.
    :param ref: MatrixTable with columns of desired ordering.
    
    """
    assert list(mt.col) == list(ref.col), 'expects all datasets to have the same columns.'
    assert list(mt.row) == list(mt.row), 'expects all datasets to have the same rows.'
    
    mt_tmp = mt.annotate_cols(idx_in_ref = ref.add_col_index().index_cols(mt.s).col_idx)
    mt_tmp = mt_tmp.add_col_index()
    indices_of_ref_samples_in_mt = mt_tmp.aggregate_cols(
    hl.map(lambda x: x.col_idx,
        hl.sorted(hl.agg.collect(mt_tmp.col.select('idx_in_ref', 'col_idx')),
              key=lambda x: x.idx_in_ref)))
    mt = mt.choose_cols(indices_of_ref_samples_in_mt)    
    return mt

In [12]:
ht.markers

NameError: name 'ht' is not defined

In [14]:
imp.locus.show()

2021-12-09 15:46:27 Hail: INFO: Coerced sorted dataset


,
locus,alleles
locus<GRCh38>,array<str>
chr21:8522406,"[""G"",""A""]"


In [35]:
locus = hl.delimit([imp.locus.contig, hl.str(imp.locus.position)],':')
alleles = hl.delimit([imp.alleles[0],imp.alleles[1]],'/')
variant = hl.delimit([locus, alleles],'_')
saige_variants = ",".join(variant.collect())

#hl.delimit([imp.locus.contig, hl.str(imp.locus.position),imp.alleles[0], imp.alleles[1]],':').show()

2021-12-09 15:51:22 Hail: INFO: Coerced sorted dataset


'chr21:8522406_G/A,chr21:8522412_C/A'

# Code in github

In [15]:
path = 'data/conditional/common/spa_conditional/211111_spa_conditional_ptv_damaging_missense_Standing_height_conditioning_markers.txt'
ht = hl.import_table(path, no_header = True, delimiter = ' ')

2021-12-13 14:28:50 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type str (not specified)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type str (not specified)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type str (not specified)
  Loading field 'f8' as type str (not specified)
  Loading field 'f9' as type str (not specified)
  Loading field 'f10' as type str (not specified)
  Loading field 'f11' as type str (not specified)
  Loading field 'f12' as type str (not specified)
  Loading field 'f13' as type str (not specified)
  Loading field 'f14' as type str (not specified)
  Loading field 'f15' as type str (not specified)
  Loading field 'f16' as type str (not specified)
  Loading field 'f17' as type str (not specified)
  Loading field 'f18' as type str (not s

In [16]:
chrom = 'chr9'
ht = ht.annotate(variant = hl.delimit([ht.f0, ht.f1, ht.f3, ht.f4], ':'))
ht = ht.key_by(**hl.parse_variant(ht.variant, reference_genome = 'GRCh38'))
ht = ht.filter(hl.literal(chrom).contains(ht.locus.contig))
rows = int(ht.count())

In [17]:
wes = hl.read_matrix_table('derived/knockouts/211206/ukb_wes_200k_maf0_0.01_chr9_ptv_damaging_missense_ko.mt') # GRCh38

In [18]:
chromosomes = [x.replace('chr', '') for x in list(set(ht.locus.contig.collect()))]

2021-12-13 14:28:51 Hail: INFO: Coerced sorted dataset


In [20]:
imp = genotypes.get_ukb_imputed_v3_bgen(chromosomes)
imp = variants.liftover(imp)
imp = imp.filter_rows(hl.is_defined(ht[imp.row_key]))
imp = imp.annotate_entries(DS = imp.dosage)
imp = imp.drop('varid','new_locus','new_alleles','old_locus')
imp = imp.select_entries(imp.DS)

# load knockout file
#wes = hl.read_matrix_table(input_mt)
wes = wes.drop('eur')
n_before = wes.count()
print(f"Pre-merging variants/samples {n_before}")

# Get intersecting individuals
wes_sids = wes.s.collect()
imp_sids = imp.s.collect()
overlap = list(set(wes_sids) & set(imp_sids))[1:5]
wes = wes.filter_cols(hl.literal(set(overlap)).contains(wes.s))
imp = imp.filter_cols(hl.literal(set(overlap)).contains(imp.s))

2021-12-13 14:29:21 Hail: INFO: Number of BGEN files parsed: 1
2021-12-13 14:29:21 Hail: INFO: Number of samples in BGEN files: 487409
2021-12-13 14:29:21 Hail: INFO: Number of variants across all BGEN files: 4066774


Pre-merging variants/samples (695, 176929)


In [ ]:
# combine the variants
imp = tables.order_cols(imp, wes)
mt = imp.union_rows(wes)
n_after = mt.count()
print(f"Pre-merging variants/samples {n_after}")

In [21]:
wes.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'DS': float64
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [22]:
imp.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'DS': float64
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [ ]:
imp.DS.show()

In [ ]:
wes.DS.show()